In [ ]:
####################################################################
# Exercise 1
####################################################################
# PROBLEM:
# Analyze short-term residential real estate market trends in Bengaluru using NumPy for numerical analysis and Pandas for data management, then create a comprehensive visualization dashboard.

# TASKS:

# 1. Create Real Estate Dataset (90 days):
#    - Date: Daily dates for 90 consecutive days
#    - Price_High: Random floats ₹80–₹180 lakh (highest listed price per day)
#    - Price_Low: Random floats ₹40–₹95 lakh (always < Price_High)
#    - Avg_Price_per_sqft: Random floats ₹4,000–₹15,000
#    - Units_Listed: Random integers 30–220 per day
#    - Units_Sold: Random integers 5–120 per day (always ≤ Units_Listed)
#    - Home_Loan_Rate: Random floats 8.2–9.8 (%)

#    Store in Pandas DataFrame

# 2. Feature Engineering with Pandas:
#    - Add 'Price_Range' column (High - Low)
#    - Add 'Price_Avg' column ((High + Low) / 2)
#    - Add 'Month' column extracted from Date
#    - Add 'High_Demand_Day' boolean column (True if Units_Sold / Units_Listed ≥ 0.6)
#    - Add 'Market_Segment' based on avg price and demand:
#      * 'Premium_High_Demand': avg > ₹120L and high demand
#      * 'Premium_Low_Demand': avg > ₹120L and low demand
#      * 'Mid_Market_High_Demand': ₹70L–₹120L and high demand
#      * 'Mid_Market_Low_Demand': ₹70L–₹120L and low demand
#      * 'Affordable': avg < ₹70L

# 3. NumPy Statistical Analysis:
#    - Convert Price_High, Price_Low, Units_Listed, Units_Sold to 2D NumPy array
#    - Calculate correlation matrix between all variables
#    - Compute 7-day rolling mean for average prices (use NumPy)
#    - Identify price anomalies (days where high price > mean + 2*std)


# 4. Create Visualization Dashboard:
#    - Plot 1: Line plot of daily high/low prices with trend line (Matplotlib)
#    - Plot 2: Bar chart of total monthly units sold (Matplotlib)
#    - Plot 3: Scatter plot of Avg_Price_per_sqft vs Units_Sold colored by High_Demand_Day (Seaborn)
#    - Plot 4: Histogram of price range distribution (Matplotlib)
#    - Plot 5: Box plot of average prices by month (Seaborn)
#    - Plot 6: Count plot of market segments (Seaborn)
#    - Plot 7: Violin plot of units sold by market segment (Seaborn)


# Your code here:


In [20]:
# Create Real Estate Dataset (90 days):
#    - Date: Daily dates for 90 consecutive days
#    - Price_High: Random floats ₹80–₹180 lakh (highest listed price per day)
#    - Price_Low: Random floats ₹40–₹95 lakh (always < Price_High)
#    - Avg_Price_per_sqft: Random floats ₹4,000–₹15,000
#    - Units_Listed: Random integers 30–220 per day
#    - Units_Sold: Random integers 5–120 per day (always ≤ Units_Listed)
#    - Home_Loan_Rate: Random floats 8.2–9.8 (%)

from datetime import datetime, timedelta
import  random
import pandas as pd
def generateDataset(days = 90):
    startDate = datetime(2024,1,1)
    dateli =[] 
    for i in range(days+1):
        li=[]
        current_date  = (startDate + timedelta(days=i) ).strftime("%Y-%m-%d") 
        li.append(current_date) 
        highprice = round(random.uniform(8000000 , 18000000),2)
        li.append(highprice) 
        lowprice = round(random.uniform(4000000 , 9500000),2)
        li.append(lowprice)
        avg = round(random.uniform(4000 , 15000),2)
        li.append(lowprice)
        ul = random.randint(30 , 220)
        li.append(ul)
        us = random.randint(2 , 120)
        li.append(us)
        loan = round(random.uniform(8.2 , 9.8),2)
        li.append(loan)

        dateli.append(li)
         
    return dateli

Raw = generateDataset()
# print(Raw)

Rawdata = pd.DataFrame(Raw , columns=["Date","HighPrice","LowPrice","SqftAvg","UnitListed","UnitSold","HomeLoanRate"])
print(Rawdata)


          Date    HighPrice    LowPrice     SqftAvg  UnitListed  UnitSold  \
0   2024-01-01  16009860.12  5239405.08  5239405.08         165       120   
1   2024-01-02  12667972.30  6764824.39  6764824.39         122        27   
2   2024-01-03   9331318.02  9164749.11  9164749.11          78        70   
3   2024-01-04  16089923.89  9125167.74  9125167.74          76        26   
4   2024-01-05  17264612.86  6493819.58  6493819.58         121        89   
..         ...          ...         ...         ...         ...       ...   
86  2024-03-27  12205360.90  8287670.23  8287670.23         134        71   
87  2024-03-28   9055769.63  7929852.27  7929852.27         171        72   
88  2024-03-29   8157114.00  6390027.59  6390027.59         123        67   
89  2024-03-30  11966692.87  5723859.74  5723859.74         139         4   
90  2024-03-31  16021836.52  4023187.99  4023187.99         175        39   

    HomeLoanRate  
0           8.90  
1           9.04  
2           9.10  

In [ ]:
#2. Feature Engineering with Pandas:
#    - Add 'Price_Range' column (High - Low)
#    - Add 'Price_Avg' column ((High + Low) / 2)
#    - Add 'Month' column extracted from Date
#    - Add 'High_Demand_Day' boolean column (True if Units_Sold / Units_Listed ≥ 0.6)
#    - Add 'Market_Segment' based on avg price and demand:
#      * 'Premium_High_Demand': avg > ₹120L and high demand
#      * 'Premium_Low_Demand': avg > ₹120L and low demand
#      * 'Mid_Market_High_Demand': ₹70L–₹120L and high demand
#      * 'Mid_Market_Low_Demand': ₹70L–₹120L and low demand
#      * 'Affordable': avg < ₹70L
import numpy as np
Rawdata["PriceRange"] = Rawdata["HighPrice"] - Rawdata["LowPrice"]

Rawdata = Rawdata.assign( AvgPrice = round(((Rawdata["HighPrice"] + Rawdata["LowPrice"]) /2) ,2)
                          , Month = pd.to_datetime(Rawdata["Date"]).dt.month_name()
                          ,HighDemandDay = round(((Rawdata["UnitSold"] + Rawdata["UnitListed"]) /2) ,2)>=6.0
                          )



conditions = [
    (Rawdata["AvgPrice"] > 120) & (Rawdata["HighDemandDay"] == True),
    (Rawdata["AvgPrice"] > 120) & (Rawdata["HighDemandDay"] == False),

    (Rawdata["AvgPrice"].between(70, 120)) & (Rawdata["HighDemandDay"] == True),
    (Rawdata["AvgPrice"].between(70, 120)) & (Rawdata["HighDemandDay"] == False),

    (Rawdata["AvgPrice"] < 70)
]

choices = [
    "Premium_High_Demand",
    "Premium_Low_Demand",
    "Mid_Market_High_Demand",
    "Mid_Market_Low_Demand",
    "Affordable"
]
Rawdata["MarketSegment"] = np.select(conditions, choices,
    default="Unknown")



Rawdata

In [ ]:
#3. NumPy Statistical Analysis:
#    - Convert Price_High, Price_Low, Units_Listed, Units_Sold to 2D NumPy array
#    - Calculate correlation matrix between all variables
#    - Compute 7-day rolling mean for average prices (use NumPy)
#    - Identify price anomalies (days where high price > mean + 2*std)
